# Render Serial Recall Fitting

Orchestrator: dispatches `templates/serial_recall_fitting.ipynb` via papermill for each confusable model configuration.

In [ ]:
import papermill as pm
from cru_to_cmr.helpers import find_project_root
from cru_to_cmr.config import serial_base_configs, serial_compterm_configs, BASE_PARAMS

ROOT = find_project_root()

In [ ]:
# Shared parameters
shared_params = {
    "data_name": "Gordon2021",
    "data_query": "data['condition'] == 2",
    "data_path": f"{ROOT}/data/Gordon2021.h5",
    "base_params": BASE_PARAMS,
    "redo_fits": False,
    "redo_sims": False,
    "run_tag": "Fitting",
    "relative_tolerance": 0.001,
    "popsize": 15,
    "num_steps": 1000,
    "cross_rate": 0.9,
    "diff_w": 0.85,
    "best_of": 3,
    "experiment_count": 50,
    "seed": 0,
    "list_lengths": [5, 6, 7],
    "analysis_paths": [
        "cru_to_cmr.analyses.srac.plot_srac",
        "cru_to_cmr.analyses.intrusion_error_rate.plot_intrusion_error_rate",
        "cru_to_cmr.analyses.omission_error_rate.plot_omission_error_rate",
        "cru_to_cmr.analyses.order_error_rate.plot_order_error_rate",
        "cru_to_cmr.analyses.crp.plot_crp",
    ],
}

# Optional: set to a model name to run only that config, or None for all
target_model = None

## Dispatch all configs

In [ ]:
template = f"{ROOT}/analyses/templates/serial_recall_fitting.ipynb"
data_name = shared_params["data_name"]
run_tag = shared_params["run_tag"]
rendered_dir = f"{ROOT}/analyses/rendered"

configs = [
    ("base", serial_base_configs),
    ("compterm", serial_compterm_configs),
]

for factory_type, model_configs in configs:
    for model_name, bounds in model_configs.items():
        if target_model is not None and model_name != target_model:
            continue

        file_model_name = model_name.replace(" ", "_")
        output_path = f"{rendered_dir}/fitting_{data_name}_{file_model_name}_{run_tag}.ipynb"

        params = shared_params | {
            "model_name": model_name,
            "factory_type": factory_type,
            "bounds": bounds,
        }

        print(f"Rendering: {model_name} ({factory_type})")
        pm.execute_notebook(
            template,
            output_path,
            parameters=params,
            progress_bar=True,
        )
        print(f"  -> {output_path}")